In [2]:
import subprocess
import sys

# Required libraries for your code
libraries = [
    "numpy",
    "matplotlib",
    "tensorflow",
    "scikit-learn"
]

print("Installing required libraries...\n")

for lib in libraries:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
        print(f"{lib} installed successfully!")
    except Exception as e:
        print(f"Failed to install {lib}: {e}")

print("\nAll libraries installed successfully!")


import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score
import numpy as np

Installing required libraries...

numpy installed successfully!
matplotlib installed successfully!
tensorflow installed successfully!
scikit-learn installed successfully!

All libraries installed successfully!


In [3]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Define image dimensions and batch size
img_height, img_width = 128, 128
batch_size = 20

# Define directories for training and testing data
train_data_dir = "../dataset/train"
test_data_dir = "../dataset/test"

In [5]:
# Data augmentation for training images
train_datagen = ImageDataGenerator(
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rescale=1./255  # Normalize pixel values
)

# Data augmentation for testing images (only rescale)
test_datagen = ImageDataGenerator(rescale=1./255)


In [6]:

# Create data generators for training and testing
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_height, img_width),
    batch_size=5,
    class_mode='categorical',
    shuffle=False  # No need to shuffle for evaluation
)

Found 1174 images belonging to 14 classes.
Found 428 images belonging to 14 classes.


In [7]:

# Create the classification head
model = Sequential([
   # Convolution Layer 1
    Conv2D(32, (3, 3), input_shape=(img_height,img_width,3), activation='relu'),
    # MaxPooling2D(pool_size=(2, 2)),
    MaxPooling2D(pool_size=(2, 2)),

    # Convolution Layer 2
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Convolution Layer 3
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Flattening
    Flatten(),

    # Fully Connected Layer 1
    Dense(128, activation='relu'),
    # Dropout(0.5),

    # Fully Connected Layer 2
    Dense(64, activation='relu'),

    # Output Layer
    Dense(14, activation='sigmoid')  
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


c:\Users\chait\OneDrive\Desktop\CNS\vitamin-deficiency-main\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 14)             │           910 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,313,806 (12.64 MB)

 Trainable params: 3,313,806 (12.64 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=15,
    validation_data=test_generator,
    validation_steps=len(test_generator)
)




Epoch 1/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 122ms/step - accuracy: 0.1431 - loss: 2.4928 - val_accuracy: 0.2827 - val_loss: 2.2747
Epoch 2/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 119ms/step - accuracy: 0.2751 - loss: 2.0981 - val_accuracy: 0.4136 - val_loss: 1.6765
Epoch 3/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 120ms/step - accuracy: 0.3629 - loss: 1.7740 - val_accuracy: 0.3481 - val_loss: 1.6089
Epoch 4/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 119ms/step - accuracy: 0.4242 - loss: 1.5687 - val_accuracy: 0.4579 - val_loss: 1.4970
Epoch 5/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 116ms/step - accuracy: 0.5221 - loss: 1.3287 - val_accuracy: 0.4766 - val_loss: 1.4902
Epoch 6/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 113ms/step - accuracy: 0.5596 - loss: 1.1891 - val_accuracy: 0.5374 - val_loss: 1.3685
Epoch 7/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 116ms/step - accuracy: 0.6150 - loss: 1.0571 - val_accuracy: 0.4836 - val_loss: 1.4144
Epoch 8/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 117ms/step - accuracy: 0.6422 - loss: 1.0189 - val_accuracy: 0.

In [9]:
# predicting train dataset 
test_pridiction = model.predict(test_generator)

# true labels 
test_true_labels = test_generator.classes




86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


In [10]:
print( test_pridiction.shape)
print( test_true_labels.shape)




(428, 14)
(428,)


In [11]:
#calculating accuracy
accuracy = accuracy_score(test_true_labels, np.argmax(test_pridiction, axis=1))
print(accuracy)

0.6378504672897196


In [ ]:
# Save the trained model
model.save("../model_saved_files/Cnn.h5")
print("Model saved successfully!")

Model saved successfully!
